# MGMTMSA 403
# Assignment 1: Operating Room Scheduling
## Group Member: Man Mei (106539885), Fuchun Yang (406547488), Yuqi Gu (506539826)

### a)

In [ ]:
pip install gurobipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.4/14.4 MB 39.4 MB/s eta 0:00:00


In [ ]:
import gurobipy as gp
from gurobipy import *

In [ ]:
# Define model and parameters
mod = Model()

I = 6  # Departments
J = 5  # Rooms
K = 5  # Days

# Parameters
c = [0.484, 0.042, 0.253, 0.074, 0.053, 0.095]  # Time share
t = [213.5 * ci for ci in c]  # Target hours
r = [  # Room availability matrix
    [9, 9, 9, 9, 7.5], # Room 1
    [9, 9, 9, 9, 7.5], # ...
    [9, 9, 9, 9, 7.5], # ...
    [9, 9, 9, 9, 7.5], # ...
    [9, 8, 8, 8, 6.5]  # Room 5
]

#print(t) # check if t is calculated correct or not

Restricted license - for non-production use only - expires 2026-11-23


In [ ]:
# Decision Variables
x = mod.addVars(I, J, K, vtype=GRB.BINARY, name="x")  # Room assignment
s = mod.addVars(I, vtype=GRB.CONTINUOUS, lb=0, name="s")  # Under-allocation


# Constraints
# 1. Time allocation constraints
for i in range(I):
    mod.addConstr(
        quicksum(x[i, j, k] * r[j][k] for j in range(J) for k in range(K)) + s[i] >= t[i],
        name=f"time_allocation_{i}"
    )

# 2. Room usage constraints
for j in range(J):
    for k in range(K):
        mod.addConstr(
            quicksum(x[i, j, k] for i in range(I)) <= 1,
            name=f"room_usage_{j}_{k}"
        )

# Objective Function
mod.setObjective(quicksum(s[i] / t[i] for i in range(I)), GRB.MINIMIZE)
mod.optimize()

Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (linux64 - "Ubuntu 22.04.3 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 31 rows, 468 columns and 306 nonzeros
Model fingerprint: 0x84fcea2e
Variable types: 18 continuous, 450 integer (450 binary)
Coefficient statistics:
  Matrix range     [1e+00, 9e+00]
  Objective range  [1e-02, 1e-01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+02]
Found heuristic solution: objective 6.0000000
Found heuristic solution: objective 6.0000000
Presolve removed 0 rows and 312 columns
Presolve time: 0.00s
Presolved: 31 rows, 156 columns, 306 nonzeros
Variable types: 4 continuous, 152 integer (150 binary)
Found heuristic solution: objective 5.0000000

Root relaxation: objective 2.066116e-03, 57 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Wo

In [ ]:
if mod.status == GRB.OPTIMAL:
    print("\nOptimal Solution Found:")
    print(f"Objective Value (Total Under-Allocation): {mod.objVal}")
    for i in range(I):
        print(f"Department{i+1}: Under-Allocation(s[{i}]) = {s[i].x}")
    for i in range(I):
        for j in range(J):
            for k in range(K):
                if x[i,j,k].x > 0.5:
                    print(f"Department {i+1} assigned to Room {j+1} on Day {k + 1}")
else:
    print("No feasible solution found.")


Optimal Solution Found:
Objective Value (Total Under-Allocation): 0.051905976357871676
Department1: Under-Allocation(s[0]) = 5.333999987051263
Department2: Under-Allocation(s[1]) = 0.0
Department3: Under-Allocation(s[2]) = 0.01550000000000283
Department4: Under-Allocation(s[3]) = 0.0
Department5: Under-Allocation(s[4]) = 0.0
Department6: Under-Allocation(s[5]) = 0.0
Department 1 assigned to Room 1 on Day 2
Department 1 assigned to Room 1 on Day 3
Department 1 assigned to Room 1 on Day 4
Department 1 assigned to Room 2 on Day 2
Department 1 assigned to Room 2 on Day 3
Department 1 assigned to Room 3 on Day 2
Department 1 assigned to Room 3 on Day 3
Department 1 assigned to Room 4 on Day 1
Department 1 assigned to Room 4 on Day 2
Department 1 assigned to Room 5 on Day 1
Department 1 assigned to Room 5 on Day 3
Department 2 assigned to Room 2 on Day 4
Department 3 assigned to Room 1 on Day 1
Department 3 assigned to Room 2 on Day 1
Department 3 assigned to Room 3 on Day 1
Department 3 as

**Optimal total under-allocation**: 5.19% of the total target operating time across all departments.

### b)

In [ ]:
L = 3 # Floors
y = mod.addVars(I, L, K, vtype=GRB.BINARY, name="y")  # Floor assignment

# Room-to-floor mapping
floors = {0: 0, 1: 0, 2: 1, 3: 1, 4: 2}

# 3. Floor assignment constraints
for i in range(I):  # department
    for k in range(K):  # day
        mod.addConstr(
            2 * y[i, 0, k] >= x[i, 0, k] + x[i, 1, k],  # Floor 1
            name=f"floor1_assignment_{i}_{k}"
        )

for i in range(I):
    for k in range(K):
        mod.addConstr(
            2 * y[i, 1, k] >= x[i, 2, k] + x[i, 3, k],  # Floor 2
            name=f"floor2_assignment_{i}_{k}"
        )

for i in range(I):
    for k in range(K):
        mod.addConstr(
            y[i, 2, k] >= x[i, 4, k],  # Floor 3
            name=f"floor3_assignment_{i}_{k}"
        )

# 4. No floor split constraint
for i in range(I):
    for k in range(K):
        mod.addConstr(
            quicksum(y[i, l, k] for l in range(L)) <= 1,
            name=f"no_floor_split_{i}_{k}"
        )

# Solve model
mod.optimize()

Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (linux64 - "Ubuntu 22.04.3 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 151 rows, 558 columns and 636 nonzeros
Model fingerprint: 0x0cf5a771
Variable types: 18 continuous, 540 integer (540 binary)
Coefficient statistics:
  Matrix range     [1e+00, 9e+00]
  Objective range  [1e-02, 1e-01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+02]

MIP start from previous solve did not produce a new incumbent solution

Found heuristic solution: objective 6.0000000
Found heuristic solution: objective 6.0000000
Presolve removed 60 rows and 372 columns
Presolve time: 0.00s
Presolved: 91 rows, 186 columns, 516 nonzeros
Variable types: 4 continuous, 182 integer (180 binary)

Root relaxation: objective 1.580699e-01, 107 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     

In [ ]:
if mod.status == GRB.OPTIMAL:
    print("\nOptimal Solution Found:")
    print(f"Objective Value (Total Under-Allocation): {mod.objVal}")
    for i in range(I):
        print(f"Department {i+1}: Under-Allocation(s[{i}]) = {s[i].x}")
    for i in range(I):
        for j in range(J):
            for k in range(K):
                if x[i,j,k].x > 0.5:
                    print(f"Department {i + 1} assigned to Room {j + 1} on Day {k + 1}")
else:
    print("No feasible solution found.")


Optimal Solution Found:
Objective Value (Total Under-Allocation): 0.15806994793581977
Department 1: Under-Allocation(s[0]) = 16.334
Department 2: Under-Allocation(s[1]) = 0.0
Department 3: Under-Allocation(s[2]) = 0.0
Department 4: Under-Allocation(s[3]) = 0.0
Department 5: Under-Allocation(s[4]) = 0.0
Department 6: Under-Allocation(s[5]) = 0.0
Department 1 assigned to Room 3 on Day 1
Department 1 assigned to Room 3 on Day 2
Department 1 assigned to Room 3 on Day 3
Department 1 assigned to Room 3 on Day 4
Department 1 assigned to Room 3 on Day 5
Department 1 assigned to Room 4 on Day 1
Department 1 assigned to Room 4 on Day 2
Department 1 assigned to Room 4 on Day 3
Department 1 assigned to Room 4 on Day 4
Department 1 assigned to Room 4 on Day 5
Department 2 assigned to Room 2 on Day 3
Department 3 assigned to Room 1 on Day 2
Department 3 assigned to Room 1 on Day 4
Department 3 assigned to Room 1 on Day 5
Department 3 assigned to Room 2 on Day 4
Department 3 assigned to Room 2 on Da

**Optimal total under-allocation**: 15.807% of the total target operating time across all departments.